# Qwen-27B Right-Side Error-Specific Handoff

The large behavior-only mirror screen found 14 pairs that Qwen-27B answered correctly in both orientations and 69 pairs for which it answered the left orientation correctly but the right orientation incorrectly.

[实验设置与运行说明](../../../docs/experiment_notes/11_handoff_development_and_matching/630450d67f2aeea2.md)


In [ ]:
# Run once. Every later code cell reports both cell and inner-loop progress.
import re
import threading
import time
from IPython.core.magic import register_cell_magic
from tqdm.auto import tqdm

_SETUP_PROGRESS = tqdm(total=1, desc='Initialize progress display', unit='cell')
_ACTIVE_CELL_PROGRESS = None

def cell_stage(message):
    if _ACTIVE_CELL_PROGRESS is not None:
        _ACTIVE_CELL_PROGRESS['stage'] = str(message)
        elapsed = time.monotonic() - _ACTIVE_CELL_PROGRESS['started']
        _ACTIVE_CELL_PROGRESS['bar'].set_postfix_str(
            f"{message} | elapsed {elapsed:.0f}s", refresh=True
        )

@register_cell_magic
def cell_progress(line, cell):
    global _ACTIVE_CELL_PROGRESS
    specification = line.strip() or 'Cell progress'
    match = re.fullmatch(r'(\d+)/(\d+)\s+(.+)', specification)
    if match:
        current, total, label = int(match.group(1)), int(match.group(2)), match.group(3)
        initial = current - 1
    else:
        current, total, label, initial = 1, 1, specification, 0
    started = time.monotonic()
    bar = tqdm(
        total=total, initial=initial, desc=label, unit='cell',
        dynamic_ncols=True, leave=True,
    )
    state = {'bar': bar, 'started': started, 'stage': 'starting'}
    _ACTIVE_CELL_PROGRESS = state
    stopped = threading.Event()

    def heartbeat():
        while not stopped.wait(1.0):
            elapsed = time.monotonic() - started
            bar.set_postfix_str(
                f"{state['stage']} | elapsed {elapsed:.0f}s", refresh=True
            )

    worker = threading.Thread(target=heartbeat, daemon=True)
    worker.start()
    succeeded = False
    try:
        cell_stage('running')
        result = get_ipython().run_cell(cell)
        result.raise_error()
        succeeded = True
        return result.result
    finally:
        stopped.set()
        worker.join(timeout=2.0)
        elapsed = time.monotonic() - started
        if succeeded:
            bar.n = current
            bar.set_postfix_str(f"complete | elapsed {elapsed:.0f}s", refresh=True)
        else:
            bar.set_postfix_str(f"failed | elapsed {elapsed:.0f}s", refresh=True)
        bar.close()
        _ACTIVE_CELL_PROGRESS = None

_SETUP_PROGRESS.update(1)
_SETUP_PROGRESS.close()
print('Cell progress helper ready: 17 tracked cells.')


## 1. Frozen route and behavior-defined matched cohorts


In [ ]:
%%cell_progress 1/17 Mount Drive and locate VisualTorque
cell_stage('Mounting Google Drive')
from google.colab import drive
drive.mount('/content/drive')

cell_stage('Locating VisualTorque project')
from pathlib import Path
PROJECT_CANDIDATES = (
    Path('/content/drive/MyDrive/VisualTorque'),
    Path('/content/drive/MyDrive/VisualTorque/VisualTorque'),
)
PROJECT_ROOT = next(
    (
        path for path in PROJECT_CANDIDATES
        if (path / 'toolkit' / 'pyproject.toml').is_file()
    ),
    None,
)
assert PROJECT_ROOT is not None, (
    'Could not find VisualTorque/toolkit/pyproject.toml on Drive.'
)
TOOLKIT_ROOT = PROJECT_ROOT / 'toolkit'
print('Project:', PROJECT_ROOT)


In [ ]:
%%cell_progress 2/17 Install and verify the multimodal environment
import importlib
import json
import os
import signal
import subprocess
import sys

cell_stage('Removing unused TorchAudio')
subprocess.run(
    [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchaudio'],
    check=False,
)

ANALYSIS_STACK = {
    'numpy': '2.5.1',
    'scipy': '1.18.0',
    'pandas': '3.0.5',
    'matplotlib': '3.11.1',
    'scikit-learn': '1.9.0',
    'tqdm': '4.67.1',
}
ENV_MARKER = Path('/content/visualtorque_inference_analysis_stack_v3.json')
probe_code = '''
import json
import matplotlib
import numpy
import pandas
import scipy
import sklearn
import tqdm
print(json.dumps({
    'numpy': numpy.__version__,
    'scipy': scipy.__version__,
    'pandas': pandas.__version__,
    'matplotlib': matplotlib.__version__,
    'scikit-learn': sklearn.__version__,
    'tqdm': tqdm.__version__,
}, sort_keys=True))
'''

def fresh_process_probe():
    return subprocess.run(
        [sys.executable, '-c', probe_code],
        text=True, capture_output=True, check=False,
    )

try:
    marker = json.loads(ENV_MARKER.read_text(encoding='utf-8'))
except (FileNotFoundError, json.JSONDecodeError):
    marker = None
probe = fresh_process_probe()
stack_ready = (
    marker == ANALYSIS_STACK
    and probe.returncode == 0
    and json.loads(probe.stdout.strip().splitlines()[-1]) == ANALYSIS_STACK
)
if not stack_ready:
    cell_stage('Repairing the pinned binary analysis stack')
    packages = [
        f'{name}=={version}' for name, version in ANALYSIS_STACK.items()
    ]
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install',
        '--no-cache-dir', '--force-reinstall', '--only-binary=:all:',
        *packages,
    ])
    repaired = fresh_process_probe()
    if repaired.returncode != 0:
        raise RuntimeError(
            'The repaired stack still fails in a fresh Python process.\n'
            + repaired.stderr[-4000:]
        )
    repaired_versions = json.loads(
        repaired.stdout.strip().splitlines()[-1]
    )
    assert repaired_versions == ANALYSIS_STACK, repaired_versions
    ENV_MARKER.write_text(
        json.dumps(ANALYSIS_STACK, sort_keys=True), encoding='utf-8'
    )
    print(
        '\nPinned binary packages were repaired. Colab will restart once. '
        'After it reconnects, choose Runtime > Run all. This is expected.',
        flush=True,
    )
    os.kill(os.getpid(), signal.SIGKILL)

cell_stage('Installing pinned inference dependencies')
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '-U',
    'transformers==5.14.0', 'accelerate>=1.10', 'sentencepiece',
    'protobuf', 'huggingface_hub>=0.34',
])
verified = fresh_process_probe()
assert verified.returncode == 0, verified.stderr[-4000:]
assert json.loads(verified.stdout.strip().splitlines()[-1]) == ANALYSIS_STACK

cell_stage('Registering the live VisualTorque source tree')
SRC_ROOT = TOOLKIT_ROOT / 'src'
PACKAGE_ROOT = SRC_ROOT / 'visual_torque_toolkit'
assert (PACKAGE_ROOT / '__init__.py').is_file(), PACKAGE_ROOT
source_text = str(SRC_ROOT)
sys.path[:] = [entry for entry in sys.path if entry != source_text]
sys.path.insert(0, source_text)
for module_name in tuple(sys.modules):
    if (
        module_name == 'visual_torque_toolkit'
        or module_name.startswith('visual_torque_toolkit.')
    ):
        del sys.modules[module_name]
importlib.invalidate_caches()

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import sklearn
import torch
import torchvision
import visual_torque_toolkit
from IPython.display import display
from transformers import AutoModelForMultimodalLM, AutoProcessor
from visual_torque_toolkit.behavior.adapter import HuggingFaceMultimodalAdapter

required_features = (
    '_template', '_to_device', '_candidate_plan', '_append_text_prefix',
)
assert all(
    hasattr(HuggingFaceMultimodalAdapter, name)
    for name in required_features
)
toolkit_path = Path(visual_torque_toolkit.__file__).resolve()
assert PACKAGE_ROOT.resolve() in toolkit_path.parents, toolkit_path
print('torch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('numpy / scipy / sklearn:',
      np.__version__, scipy.__version__, sklearn.__version__)
print('Toolkit source:', toolkit_path)


In [ ]:
%%cell_progress 3/17 Verify GPU and Hugging Face access
cell_stage('Checking CUDA device')
assert torch.cuda.is_available(), 'Select an A100 GPU runtime.'
print(torch.cuda.get_device_name(0))
print('CUDA memory (GB):', round(
    torch.cuda.get_device_properties(0).total_memory / 2**30, 1
))

cell_stage('Authenticating with Hugging Face')
from google.colab import userdata
from huggingface_hub import login
try:
    token = userdata.get('HF_TOKEN')
    if token:
        login(token=token, add_to_git_credential=False)
        print('Hugging Face login complete.')
    else:
        print('HF_TOKEN is not set. Public checkpoints may still work.')
except Exception as exc:
    print('Hugging Face login skipped:', exc)


In [ ]:
%%cell_progress 4/17 Lock the matched error-specific protocol
cell_stage('Freezing cohorts, matching, route, directions, doses, and gates')
import hashlib
import json
from datetime import datetime, timezone

MODEL_KEY = 'qwen27b'
MODEL_ID = 'Qwen/Qwen3.5-27B'
MODEL_SLUG = 'Qwen__Qwen3.5-27B__373612cb'
MODEL_REVISION = '373612cb4f9288fcf550b526d86d85febedd9801'
MODEL_DTYPE = 'bfloat16'
DEVICE_MAP = 'auto'
EXPECTED_LAYER_COUNT = 64
UPSTREAM_LAYER = 48
UPSTREAM_METHOD = 'pls4_prototype'
SEMANTIC_LAYER = 49
LABEL_LAYER = 55
CAPTURE_LAYERS = (UPSTREAM_LAYER, SEMANTIC_LAYER, LABEL_LAYER)
SITE_SCOPE = 'final_prompt_token'

SEED = 20261115
READINESS_SEED = 20261112
DIRECTION_SEED = 20260901
LABELS = ('A', 'B', 'C')
MAPPING_VARIANTS = ('forward', 'reverse')
DONOR_TASKS = ('weight_more', 'weight_less')
QUESTION_KINDS = (
    'outcome', 'left_units', 'right_units', 'left_distance',
    'right_distance', 'left_torque', 'right_torque', 'net_torque',
)
COHORTS = ('both_correct', 'right_error')
SIDES = ('left', 'right')
MATCHED_UNITS = 14
BOOTSTRAP_REPLICATES = 10000
ALPHA_LEVEL = 0.05
EQUIVALENCE_FRACTION = 0.25
MEDIATOR_MATCH_FRACTION = 0.20
MATCHING_MAX_SMD = 0.50
BASELINE_REPLAY_MIN_CONCORDANCE = 0.90
BASELINE_REPLAY_MIN_CELL_CONCORDANCE = 0.80
PATCH_CONDITION_BATCH_SIZE = 1
SMOKE_CENTERED_LOGPROB_TOLERANCE = 0.15
NORM_RELATIVE_TOLERANCE = 0.02

PARENT_ROOT = (
    PROJECT_ROOT / 'results'
    / 'qwen27b_nettorque_semantic_handoff_mediation_v1'
)
PARENT_RUN_PATH = PARENT_ROOT / 'RUN_COMPLETE.json'
PARENT_ROUTE_PATH = PARENT_ROOT / 'selected_upstream_route.json'
PARENT_DIRECTION_PATH = PARENT_ROOT / 'direction_library.json'
PARENT_ESTIMANDS_PATH = PARENT_ROOT / 'analysis' / 'mapping_averaged_estimands.csv'

READINESS_ROOT = (
    PROJECT_ROOT / 'results' / 'side_correctness_2x2_readiness_v3'
)
READINESS_RUN_PATH = READINESS_ROOT / 'RUN_COMPLETE.json'
READINESS_PROTOCOL_PATH = READINESS_ROOT / 'frozen_protocol.json'
READINESS_CANDIDATE_PATH = READINESS_ROOT / 'paired_candidates.jsonl'
READINESS_SCREEN_PATH = READINESS_ROOT / 'screen_records.jsonl'
READINESS_PAIR_TABLE_PATH = READINESS_ROOT / 'pair_readiness.csv'
READINESS_MIRROR_DIR = READINESS_ROOT / 'mirrored_images'

AUDIT_ROOT = (
    PROJECT_ROOT / 'results'
    / 'qwen27b_error_conditioned_physical_state_audit_v1'
)
AUDIT_OUTCOME_MANIFEST_PATH = AUDIT_ROOT / 'outcome_prompt_manifest.jsonl'
AUDIT_OUTCOME_SHARD_DIR = AUDIT_ROOT / 'outcome_capture_shards'
DEPTH_SUPPORT_PATH = (
    PROJECT_ROOT / 'results' / 'semantic_label_depth_localization_v1'
    / 'depth_support.jsonl'
)
INVERSE_SCENE_DIR = (
    PROJECT_ROOT / 'results' / 'inverse_query_weight_route_specificity_v1'
    / 'rendered_scenes'
)

OUTPUT_ROOT = (
    PROJECT_ROOT / 'results'
    / 'qwen27b_rightside_error_specific_handoff_v2'
)
PROTOCOL_LOCK_PATH = OUTPUT_ROOT / 'frozen_protocol.json'
MATCH_MANIFEST_PATH = OUTPUT_ROOT / 'matched_pair_manifest.jsonl'
SCENE_MANIFEST_PATH = OUTPUT_ROOT / 'matched_scene_manifest.jsonl'
MATCH_AUDIT_PATH = OUTPUT_ROOT / 'matching_audit.json'
BASELINE_PATH = OUTPUT_ROOT / 'baseline_mapping_replay.jsonl'
DIRECTION_LIBRARY_PATH = OUTPUT_ROOT / 'direction_library.json'
SHARD_DIR = OUTPUT_ROOT / 'intervention_shards'
ANALYSIS_DIR = OUTPUT_ROOT / 'analysis'
FIGURE_DIR = ANALYSIS_DIR / 'figures'
for path in (OUTPUT_ROOT, SHARD_DIR, ANALYSIS_DIR, FIGURE_DIR):
    path.mkdir(parents=True, exist_ok=True)

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return 'sha256:' + digest.hexdigest()

def stable_key(value, seed=SEED):
    return hashlib.sha256(f'{seed}:{value!r}'.encode('utf-8')).hexdigest()

def json_default(value):
    if isinstance(value, np.generic):
        return value.item()
    if torch.is_tensor(value) and value.numel() == 1:
        return value.item()
    raise TypeError(type(value).__name__)

def write_json_atomic(path, value):
    temporary = Path(path).with_suffix(Path(path).suffix + '.tmp')
    temporary.write_text(json.dumps(
        value, indent=2, ensure_ascii=False, default=json_default
    ) + '\n', encoding='utf-8')
    temporary.replace(path)

def write_jsonl_atomic(path, rows):
    temporary = Path(path).with_suffix(Path(path).suffix + '.tmp')
    with temporary.open('w', encoding='utf-8') as handle:
        for row in rows:
            handle.write(json.dumps(
                row, sort_keys=True, default=json_default
            ) + '\n')
    temporary.replace(path)

def read_jsonl(path):
    with Path(path).open('r', encoding='utf-8') as handle:
        return [json.loads(line) for line in handle if line.strip()]

required_inputs = (
    PARENT_RUN_PATH, PARENT_ROUTE_PATH, PARENT_DIRECTION_PATH,
    PARENT_ESTIMANDS_PATH, READINESS_RUN_PATH, READINESS_PROTOCOL_PATH,
    READINESS_CANDIDATE_PATH, READINESS_SCREEN_PATH,
    READINESS_PAIR_TABLE_PATH, READINESS_MIRROR_DIR,
    AUDIT_OUTCOME_MANIFEST_PATH, AUDIT_OUTCOME_SHARD_DIR,
    DEPTH_SUPPORT_PATH, INVERSE_SCENE_DIR,
)
for required in required_inputs:
    assert Path(required).exists(), f'Missing prerequisite: {required}'

readiness_run = json.loads(READINESS_RUN_PATH.read_text(encoding='utf-8'))
assert readiness_run['primary_decision'] == 'side_by_correctness_2x2_support_insufficient'
parent_route = json.loads(PARENT_ROUTE_PATH.read_text(encoding='utf-8'))
assert parent_route['selected_layer'] == UPSTREAM_LAYER
assert parent_route['selected_method'] == UPSTREAM_METHOD
parent_direction = json.loads(PARENT_DIRECTION_PATH.read_text(encoding='utf-8'))
parent_estimands = pd.read_csv(PARENT_ESTIMANDS_PATH)
upstream = parent_estimands[
    parent_estimands['condition'].eq('upstream_exact')
].groupby('alpha', as_index=False)['l49_toward_exact'].mean().sort_values('alpha')
alpha_values = upstream['alpha'].to_numpy(dtype=float)
projection_values = upstream['l49_toward_exact'].to_numpy(dtype=float)
UPSTREAM_TO_L49_SLOPE = float(
    np.dot(alpha_values, projection_values) / np.dot(alpha_values, alpha_values)
)
PARENT_SEMANTIC_NORM = float(parent_direction['semantic_norm'])
MATCHED_UPSTREAM_ALPHA = float(PARENT_SEMANTIC_NORM / UPSTREAM_TO_L49_SLOPE)
assert 2.0 < MATCHED_UPSTREAM_ALPHA < 8.0

parent_primary = parent_estimands[np.isclose(parent_estimands['alpha'], 1.0)]
parent_l49 = parent_primary[parent_primary['condition'].eq('l49_exact')]
ANSWER_EQUIVALENCE_MARGIN = float(
    EQUIVALENCE_FRACTION * abs(parent_l49['margin_shift'].mean())
)
CODE_EQUIVALENCE_MARGIN = float(
    EQUIVALENCE_FRACTION * abs(parent_l49['l55_toward_exact_code'].mean())
)
CONDITION_SCALES = {
    'upstream_natural': 1.0,
    'upstream_halfmatched': 0.5 * MATCHED_UPSTREAM_ALPHA,
    'upstream_matched': MATCHED_UPSTREAM_ALPHA,
    'upstream_onehalfmatched': 1.5 * MATCHED_UPSTREAM_ALPHA,
    'upstream_reverse_matched': MATCHED_UPSTREAM_ALPHA,
    'upstream_random_matched': MATCHED_UPSTREAM_ALPHA,
    'l49_direct': 1.0, 'l49_reverse': 1.0, 'l49_random': 1.0,
    'l55_direct': 1.0, 'l55_stale': 1.0, 'l55_random': 1.0,
}

protocol_identity = {
    'schema_version': 'rightside-error-specific-handoff-1.1',
    'analysis_status': 'behavior-defined matched partial factorial confirmation',
    'model_id': MODEL_ID, 'revision': MODEL_REVISION,
    'parent_run_sha256': sha256_file(PARENT_RUN_PATH),
    'readiness_run_sha256': sha256_file(READINESS_RUN_PATH),
    'readiness_candidate_sha256': sha256_file(READINESS_CANDIDATE_PATH),
    'readiness_screen_sha256': sha256_file(READINESS_SCREEN_PATH),
    'cohorts': {
        'both_correct': 'left and right stable-correct under both mappings',
        'right_error': 'left stable-correct and right stable-error',
    },
    'matching': (
        'all 14 both-correct pairs; Hungarian selection of 14 right-error '
        'pairs with diagnostic mismatch penalty, torque-bin penalty, and '
        'standardized orientation-neutral physical distance'
    ),
    'route': {'layer': UPSTREAM_LAYER, 'method': UPSTREAM_METHOD},
    'matched_upstream_alpha': MATCHED_UPSTREAM_ALPHA,
    'conditions': CONDITION_SCALES,
    'primary_interaction': (
        '(right-left in right_error) minus (right-left in both_correct)'
    ),
    'baseline_replay_gate': {
        'global_minimum': BASELINE_REPLAY_MIN_CONCORDANCE,
        'per_cell_minimum': BASELINE_REPLAY_MIN_CELL_CONCORDANCE,
        'reason': (
            'readiness used a two-mapping batch; intervention replay scores '
            'each mapping separately and may expose low-margin numerical flips'
        ),
    },
    'bootstrap_unit': 'matched quadruplet',
    'selection_uses_causal_results': False,
    'seed': SEED,
}
canonical_identity = json.loads(json.dumps(protocol_identity, sort_keys=True))
if PROTOCOL_LOCK_PATH.is_file():
    previous = json.loads(PROTOCOL_LOCK_PATH.read_text(encoding='utf-8'))
    assert previous.get('identity') == canonical_identity, (
        'Existing protocol differs. Use a new versioned output directory.'
    )
else:
    write_json_atomic(PROTOCOL_LOCK_PATH, {
        'identity': canonical_identity,
        'locked_at_utc': datetime.now(timezone.utc).isoformat(),
    })
print('Matched upstream alpha:', MATCHED_UPSTREAM_ALPHA)
print('Required matched quadruplets:', MATCHED_UNITS)


In [ ]:
%%cell_progress 5/17 Prepare Dataset v1 on local SSD
cell_stage('Checking local and Drive dataset caches')
from zipfile import ZipFile

REQUIRED_DATASET_FILES = {
    'dataset_config.json', 'scenes.jsonl', 'counterfactual_pairs.jsonl',
    'questions.jsonl', 'splits.json', 'statistics.json',
}
DATASET_DIR = Path('/content/visualtorque_dataset_v1')
DATASET_ARCHIVE = PROJECT_ROOT / 'data' / 'dataset_v1.zip'
LEGACY_DATASET_DIR = PROJECT_ROOT / 'data' / 'dataset_v1'

def dataset_complete(path):
    return path.is_dir() and all(
        (path / name).is_file() for name in REQUIRED_DATASET_FILES
    )

if not dataset_complete(DATASET_DIR):
    DATASET_DIR.mkdir(parents=True, exist_ok=True)
    if DATASET_ARCHIVE.is_file():
        cell_stage('Extracting Dataset v1 archive')
        with ZipFile(DATASET_ARCHIVE, 'r') as bundle:
            members = [m for m in bundle.infolist() if not m.is_dir()]
            root = DATASET_DIR.resolve()
            for member in tqdm(members, desc='Extracting Dataset v1', unit='file'):
                target = (DATASET_DIR / member.filename).resolve()
                if root not in target.parents:
                    raise ValueError(f'Unsafe archive path: {member.filename}')
                bundle.extract(member, DATASET_DIR)
    elif dataset_complete(LEGACY_DATASET_DIR):
        DATASET_DIR = LEGACY_DATASET_DIR
    else:
        raise FileNotFoundError(
            'Run the Dataset v1 build/cache cell in the behavior notebook first.'
        )
assert dataset_complete(DATASET_DIR)
print('Dataset:', DATASET_DIR)


In [ ]:
%%cell_progress 6/17 Freeze matched correct and right-error cohorts
cell_stage('Matching behavior-defined cohorts before causal inference')
from collections import Counter, defaultdict
from dataclasses import replace as dataclass_replace
import shutil
from scipy.optimize import linear_sum_assignment
from visual_torque_toolkit.behavior import (
    BehaviorRunConfig, PromptCondition, load_behavior_examples,
)

candidate_manifest = read_jsonl(READINESS_CANDIDATE_PATH)
candidate_by_scene = {str(row['scene_id']): row for row in candidate_manifest}
candidate_by_pair = defaultdict(list)
for row in candidate_manifest:
    candidate_by_pair[str(row['pair_id'])].append(row)
pair_frame = pd.read_csv(READINESS_PAIR_TABLE_PATH)
assert pair_frame['pair_id'].nunique() == 129
assert int((pair_frame['pattern'] == 'left_correct__right_correct').sum()) == 14
assert int((pair_frame['pattern'] == 'left_correct__right_error').sum()) == 69

def pair_feature_record(pair_id):
    rows = candidate_by_pair[str(pair_id)]
    anchor = next(row for row in rows if row['orientation'] == 'anchor')
    geometry = sorted(tuple(map(int, item)) for item in anchor['pair_geometry'])
    flat = [value for item in geometry for value in item]
    return {
        'pair_id': str(pair_id),
        'diagnostic_cell': str(anchor['diagnostic_cell']),
        'margin_bin': int(anchor['margin_bin']),
        'units_1': flat[0], 'distance_1': flat[1],
        'units_2': flat[2], 'distance_2': flat[3],
        'absolute_net_torque': int(anchor['absolute_net_torque']),
        'total_units': int(anchor['left_units']) + int(anchor['right_units']),
        'unit_difference': abs(int(anchor['left_units']) - int(anchor['right_units'])),
        'distance_difference': abs(
            int(anchor['left_distance']) - int(anchor['right_distance'])
        ),
    }

correct_ids = pair_frame.loc[
    pair_frame['pattern'].eq('left_correct__right_correct'), 'pair_id'
].astype(str).tolist()
error_ids = pair_frame.loc[
    pair_frame['pattern'].eq('left_correct__right_error'), 'pair_id'
].astype(str).tolist()
correct_features = pd.DataFrame([pair_feature_record(value) for value in correct_ids])
error_features = pd.DataFrame([pair_feature_record(value) for value in error_ids])
feature_columns = [
    'units_1', 'distance_1', 'units_2', 'distance_2',
    'absolute_net_torque', 'total_units', 'unit_difference',
    'distance_difference',
]
combined = pd.concat([
    correct_features[feature_columns], error_features[feature_columns]
], ignore_index=True).astype(float)
scale = combined.std(ddof=0).replace(0, 1.0)
correct_z = correct_features[feature_columns].astype(float) / scale
error_z = error_features[feature_columns].astype(float) / scale
cost = np.zeros((len(correct_features), len(error_features)), dtype=float)
for i, correct in correct_features.iterrows():
    for j, error in error_features.iterrows():
        physical = float(np.linalg.norm(
            correct_z.iloc[i].to_numpy() - error_z.iloc[j].to_numpy()
        ))
        diagnostic_penalty = 25.0 * float(
            correct['diagnostic_cell'] != error['diagnostic_cell']
        )
        margin_penalty = 3.0 * abs(
            int(correct['margin_bin']) - int(error['margin_bin'])
        )
        jitter = int(stable_key((
            'matching-jitter', correct['pair_id'], error['pair_id']
        ))[:8], 16) / 2**48
        cost[i, j] = diagnostic_penalty + margin_penalty + physical + jitter
row_indices, column_indices = linear_sum_assignment(cost)
assert len(row_indices) == MATCHED_UNITS

match_manifest = []
for match_index, (i, j) in enumerate(zip(row_indices, column_indices)):
    correct = correct_features.iloc[int(i)]
    error = error_features.iloc[int(j)]
    match_manifest.append({
        'match_id': f'match_{match_index:02d}',
        'both_correct_pair_id': str(correct['pair_id']),
        'right_error_pair_id': str(error['pair_id']),
        'matching_cost': float(cost[int(i), int(j)]),
        'diagnostic_cell_match': bool(
            correct['diagnostic_cell'] == error['diagnostic_cell']
        ),
        'margin_bin_difference': abs(
            int(correct['margin_bin']) - int(error['margin_bin'])
        ),
    })
write_jsonl_atomic(MATCH_MANIFEST_PATH, match_manifest)

selected_correct_ids = [row['both_correct_pair_id'] for row in match_manifest]
selected_error_ids = [row['right_error_pair_id'] for row in match_manifest]
selected_error_features = error_features.set_index('pair_id').loc[selected_error_ids]
selected_correct_features = correct_features.set_index('pair_id').loc[selected_correct_ids]
smd = {}
for column in feature_columns:
    left = selected_correct_features[column].to_numpy(dtype=float)
    right = selected_error_features[column].to_numpy(dtype=float)
    pooled = np.sqrt((left.var(ddof=1) + right.var(ddof=1)) / 2)
    smd[column] = float((left.mean() - right.mean()) / pooled) if pooled > 0 else 0.0
MAXIMUM_ABSOLUTE_SMD = max(abs(value) for value in smd.values())
DIAGNOSTIC_MATCH_RATE = float(np.mean([
    row['diagnostic_cell_match'] for row in match_manifest
]))
MATCHING_BALANCE_GATE = bool(
    MAXIMUM_ABSOLUTE_SMD <= MATCHING_MAX_SMD
    and DIAGNOSTIC_MATCH_RATE == 1.0
)
matching_audit = {
    'matched_units': MATCHED_UNITS,
    'diagnostic_match_rate': DIAGNOSTIC_MATCH_RATE,
    'maximum_absolute_smd': MAXIMUM_ABSOLUTE_SMD,
    'smd_by_feature': smd,
    'balance_gate': MATCHING_BALANCE_GATE,
    'causal_results_used': False,
}
write_json_atomic(MATCH_AUDIT_PATH, matching_audit)

scene_manifest = []
match_by_pair = {}
for row in match_manifest:
    match_by_pair[row['both_correct_pair_id']] = (row['match_id'], 'both_correct')
    match_by_pair[row['right_error_pair_id']] = (row['match_id'], 'right_error')
for pair_id, (match_id, cohort) in match_by_pair.items():
    for candidate in candidate_by_pair[pair_id]:
        side = str(candidate['exact_semantic'])
        baseline_status = 'error' if cohort == 'right_error' and side == 'right' else 'correct'
        scene_manifest.append({
            **candidate,
            'match_id': match_id, 'cohort': cohort, 'side': side,
            'baseline_status': baseline_status,
            'image_orientation': candidate['orientation'],
            'orientation': side,
            'source_scene_id': candidate['source_dataset_scene_id'],
        })
assert len(scene_manifest) == 4 * MATCHED_UNITS
assert Counter((row['cohort'], row['side']) for row in scene_manifest) == {
    (cohort, side): MATCHED_UNITS for cohort in COHORTS for side in SIDES
}
write_jsonl_atomic(SCENE_MANIFEST_PATH, scene_manifest)

scene_records = read_jsonl(DATASET_DIR / 'scenes.jsonl')
scene_by_id = {str(row['scene_id']): row for row in scene_records}
loader_config = BehaviorRunConfig(
    dataset_dir=DATASET_DIR, output_dir=OUTPUT_ROOT / '_unused_loader',
    model_id=MODEL_ID, model_revision=MODEL_REVISION,
    conditions=(PromptCondition.IMAGE_RULE,), question_kinds=QUESTION_KINDS,
    splits=('probe_train', 'probe_dev', 'probe_test'), batch_size=1,
    generation_every=0, seed=READINESS_SEED,
)
all_examples = tuple(load_behavior_examples(loader_config))
examples_by_scene_kind = defaultdict(list)
for example in all_examples:
    examples_by_scene_kind[(str(example.scene_id), example.question_kind)].append(example)
outcome_example_by_scene = {}
for row in scene_manifest:
    source_id = str(row['source_dataset_scene_id'])
    candidates = sorted(
        examples_by_scene_kind[(source_id, 'outcome')],
        key=lambda example: str(example.variant),
    )
    base = candidates[
        int(stable_key(('prompt-variant', source_id), seed=READINESS_SEED)[:8], 16)
        % len(candidates)
    ]
    image_path = base.image_path
    if row['image_orientation'] == 'mirror':
        image_path = READINESS_MIRROR_DIR / f"{row['scene_id']}.png"
        assert image_path.is_file(), image_path
    outcome_example_by_scene[row['scene_id']] = dataclass_replace(
        base, scene_id=row['scene_id'], image_path=Path(image_path).resolve(),
        outcome=row['exact_semantic'], net_torque=int(row['net_torque']),
    )

direction_support = read_jsonl(DEPTH_SUPPORT_PATH)
DIRECTION_RUNTIME_DIR = Path('/content/visualtorque_semantic_direction_images_v1')
DIRECTION_RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
direction_files = sorted({
    role['image_filename']
    for row in direction_support for role in row['roles'].values()
})
for filename in tqdm(direction_files, desc='Caching direction images', unit='image'):
    source = INVERSE_SCENE_DIR / filename
    destination = DIRECTION_RUNTIME_DIR / filename
    assert source.is_file(), source
    if not destination.is_file() or destination.stat().st_size != source.stat().st_size:
        shutil.copy2(source, destination)
print('Matched quadruplets / physical scenes:', MATCHED_UNITS, len(scene_manifest))
display(matching_audit)


## 2. Reconstruct the frozen mechanism and verify cohorts


In [ ]:
%%cell_progress 7/17 Load pinned Qwen-27B and decoder layers
cell_stage(f'Loading {MODEL_KEY} weights for the matched error-specific handoff test')
import gc
if 'adapter' in globals():
    del adapter
    gc.collect()
    torch.cuda.empty_cache()

adapter = HuggingFaceMultimodalAdapter(
    MODEL_ID, revision=MODEL_REVISION,
    dtype=MODEL_DTYPE, device_map=DEVICE_MAP,
)
display(adapter.metadata)

def resolve_module_path(root, path):
    value = root
    for component in path.split('.'):
        value = getattr(value, component)
    return value

def discover_decoder_layers(model):
    candidates = (
        'model.language_model.layers', 'model.text_model.layers',
        'language_model.model.layers', 'model.model.layers',
        'model.layers',
    )
    for path in candidates:
        try:
            layers = resolve_module_path(model, path)
        except AttributeError:
            continue
        if isinstance(layers, torch.nn.ModuleList) and len(layers) >= 8:
            return path, layers
    raise RuntimeError('Could not locate decoder layers.')

LAYER_PATH, DECODER_LAYERS = discover_decoder_layers(adapter.model)
assert len(DECODER_LAYERS) == EXPECTED_LAYER_COUNT
assert all(0 <= layer < len(DECODER_LAYERS) for layer in CAPTURE_LAYERS)
print('Layer path:', LAYER_PATH)
print('Capture layers:', CAPTURE_LAYERS)


In [ ]:
%%cell_progress 8/17 Define unchanged counterbalanced physical-side prompts
cell_stage('Separating physical sides from A/B/C response codes')
import math
from dataclasses import replace as dataclass_replace
from visual_torque_toolkit.behavior.data import BehaviorExample

RULE_TEXT = (
    'Each square is one unit of weight. Distance is measured in tick '
    'intervals from the center pivot. Torque equals weight times distance. '
    'Total torque on a side is the sum over stacks on that side.'
)
QUESTION_TEXT = {
    'outcome': 'Which way will the balance scale tip?',
    'weight_more': 'Which side has more total weight?',
    'weight_less': 'Which side has less total weight?',
}
SEMANTIC_TEXT = {'left': 'left side', 'balanced': 'equal', 'right': 'right side'}

def outcome(value):
    return 'left' if value > 0 else 'right' if value < 0 else 'balanced'

def task_semantic(features, task):
    if task == 'outcome':
        value = int(features['net_torque'])
    elif task == 'weight_more':
        value = int(features['weight_score'])
    elif task == 'weight_less':
        value = -int(features['weight_score'])
    else:
        raise KeyError(task)
    return outcome(value)

def mapping_by_label(semantics):
    assert set(semantics) == {'left', 'balanced', 'right'}
    return tuple({
        'position': index, 'label': LABELS[index], 'semantic': semantic,
    } for index, semantic in enumerate(semantics))

def direction_reference_mapping(row):
    semantics = ['left', 'balanced', 'right']
    offset = int(stable_key(
        f"options:{row['family_id']}", seed=DIRECTION_SEED
    )[:8], 16) % 3
    return mapping_by_label(semantics[offset:] + semantics[:offset])

def target_mappings(row):
    exact, wrong = row['exact_semantic'], row['shortcut_semantic']
    assert {exact, wrong} == {'left', 'right'}
    balanced_label = LABELS[int(stable_key(
        ('balanced-label', row['scene_id']), seed=READINESS_SEED
    )[:8], 16) % 3]
    side_labels = [label for label in LABELS if label != balanced_label]
    result = {}
    for variant, exact_label, wrong_label in (
        ('forward', side_labels[0], side_labels[1]),
        ('reverse', side_labels[1], side_labels[0]),
    ):
        by_label = {
            exact_label: exact, wrong_label: wrong, balanced_label: 'balanced'
        }
        result[variant] = mapping_by_label(tuple(by_label[label] for label in LABELS))
    return result

def semantics_by_label(mapping):
    return tuple(next(
        item['semantic'] for item in mapping if item['label'] == label
    ) for label in LABELS)

def label_for(mapping, semantic):
    return next(item['label'] for item in mapping if item['semantic'] == semantic)

def custom_prompt(task, mapping):
    options = '\n'.join(
        f"{item['label']}. {SEMANTIC_TEXT[item['semantic']]}" for item in mapping
    )
    return '\n\n'.join([
        'Answer this question using only the supplied balance-scale scene.',
        RULE_TEXT, 'Question: ' + QUESTION_TEXT[task], 'Options:\n' + options,
        'Do not explain your reasoning. Reply with exactly one option letter '
        'and nothing else. Valid replies are: A, B, C.',
    ])

def make_direction_example(row, role_record, task, mapping, name):
    features = role_record['features']
    semantics = semantics_by_label(mapping)
    gold = task_semantic(features, task)
    return BehaviorExample(
        example_id=f"error_specific_direction:{row['family_id']}:{name}",
        question_id=f"error_specific_direction_{row['family_id']}_{name}",
        scene_id=str(role_record['scene_id']), condition=PromptCondition.IMAGE_RULE,
        question_kind=task, variant='error_specific_direction', split='direction_support',
        problem_type='semantic_direction', outcome=gold,
        net_torque=int(features['net_torque']), prompt=custom_prompt(task, mapping),
        image_path=(DIRECTION_RUNTIME_DIR / role_record['image_filename']).resolve(),
        labels=LABELS, option_text=tuple(SEMANTIC_TEXT[x] for x in semantics),
        option_semantics=semantics, gold_label=LABELS[semantics.index(gold)],
        gold_semantic=gold,
    )

def build_direction_examples(row):
    mapping = direction_reference_mapping(row)
    examples = {}
    for task in DONOR_TASKS:
        for role in ('base', 'aligned_source'):
            key = f'{task}__{role}'
            examples[key] = make_direction_example(
                row, row['roles'][role], task, mapping, key
            )
    return examples, mapping

def build_target_examples(row):
    mappings = target_mappings(row)
    base = outcome_example_by_scene[row['scene_id']]
    examples = {}
    for variant, mapping in mappings.items():
        semantics = semantics_by_label(mapping)
        examples[variant] = dataclass_replace(
            base,
            example_id=f"error_specific_target:{row['scene_id']}:{variant}",
            question_id=f"error_specific_target_{row['scene_id']}_{variant}",
            variant=f'error_specific_{variant}', prompt=custom_prompt('outcome', mapping),
            labels=LABELS, option_text=tuple(SEMANTIC_TEXT[x] for x in semantics),
            option_semantics=semantics,
            gold_label=LABELS[semantics.index(row['exact_semantic'])],
            gold_semantic=row['exact_semantic'],
        )
    return examples, mappings

def output_hidden_tensor(output):
    if torch.is_tensor(output):
        hidden = output
    elif isinstance(output, (tuple, list)) and output and torch.is_tensor(output[0]):
        hidden = output[0]
    elif hasattr(output, 'last_hidden_state') and torch.is_tensor(output.last_hidden_state):
        hidden = output.last_hidden_state
    else:
        raise TypeError(f'Unsupported output: {type(output).__name__}')
    assert hidden.ndim == 3
    return hidden

def replace_output_hidden(output, modified):
    if torch.is_tensor(output):
        return modified
    if isinstance(output, tuple):
        return (modified,) + output[1:]
    if isinstance(output, list):
        return [modified] + output[1:]
    raise TypeError(type(output).__name__)

def prepare_examples(examples):
    examples = tuple(examples)
    plan = adapter._candidate_plan(examples[0])
    if plan is None:
        raise RuntimeError('A/B/C labels must be single-token candidates.')
    for example in examples[1:]:
        assert adapter._candidate_plan(example) == plan
    fixed_prefix, token_by_label = plan
    inputs = adapter._append_text_prefix(
        adapter._to_device(adapter._template(
            [example.messages() for example in examples], add_generation_prompt=True,
        )), fixed_prefix,
    )
    return {'examples': examples, 'inputs': inputs, 'token_by_label': token_by_label}

def final_mask(prepared):
    assert len(prepared['examples']) == 1
    attention = prepared['inputs']['attention_mask'][0].detach().cpu().bool()
    mask = torch.zeros_like(attention)
    mask[int(torch.where(attention)[0][-1])] = True
    return mask

print('Prompt builders ready.')


In [ ]:
%%cell_progress 9/17 Define fixed-route intervention and state scorers
cell_stage('Defining variable-layer final-token transport')

def score_records(prepared, logits):
    logprobs = torch.log_softmax(logits[:, -1, :].float(), dim=-1)
    records = []
    for index, example in enumerate(prepared['examples']):
        scores = [
            float(logprobs[index, prepared['token_by_label'][label]].detach().cpu())
            for label in example.labels
        ]
        maximum = max(scores)
        weights = [math.exp(value - maximum) for value in scores]
        probabilities = [value / sum(weights) for value in weights]
        predicted = int(np.argmax(scores))
        records.append({
            'candidate_labels': list(example.labels),
            'candidate_semantics': list(example.option_semantics),
            'candidate_logprobs': scores,
            'candidate_probabilities': probabilities,
            'predicted_label': example.labels[predicted],
            'predicted_semantic': example.option_semantics[predicted],
            'gold_label': example.gold_label,
            'gold_semantic': example.gold_semantic,
        })
    return records

def forward_capture(prepared, layers):
    hidden_by_layer = {}
    handles = []
    for layer in layers:
        def hook(_module, _inputs, output, layer=layer):
            hidden_by_layer[layer] = output_hidden_tensor(
                output
            ).detach().cpu().clone()
        handles.append(DECODER_LAYERS[layer].register_forward_hook(hook))
    try:
        with torch.inference_mode():
            output = adapter.model(**prepared['inputs'], use_cache=False)
    finally:
        for handle in handles:
            handle.remove()
    return score_records(prepared, output.logits), hidden_by_layer

def capture_examples(examples, layers):
    scores = {}
    hidden = {name: {} for name in examples}
    masks = {}
    for name, example in examples.items():
        prepared = prepare_examples([example])
        masks[name] = final_mask(prepared)
        current_scores, current_hidden = forward_capture(prepared, layers)
        scores[name] = current_scores[0]
        for layer, tensor in current_hidden.items():
            hidden[name][layer] = tensor[0]
        del prepared, current_hidden
    return scores, hidden, masks

def selected_norm(tensor):
    return float(torch.linalg.vector_norm(tensor.float()))

def matched_random(tensor, key):
    generator = torch.Generator(device='cpu')
    generator.manual_seed(int(stable_key(key)[:16], 16) % (2**63 - 1))
    noise = torch.randn(tensor.shape, generator=generator, dtype=torch.float32)
    return (noise * (selected_norm(tensor) / selected_norm(noise))).to(tensor.dtype)

def forward_intervention_chunk(prepared, conditions):
    assert len(prepared['examples']) == len(conditions)
    layers = {int(row['layer_index']) for row in conditions}
    assert len(layers) == 1
    layer = next(iter(layers))
    payloads = torch.stack([row['payload_selected'] for row in conditions])
    masks = torch.stack([row['target_mask'] for row in conditions])
    modes = [row['intervention_mode'] for row in conditions]
    counts = []
    def hook(_module, _inputs, output):
        hidden = output_hidden_tensor(output)
        payload = payloads.to(device=hidden.device, dtype=hidden.dtype)
        mask = masks.to(device=hidden.device)
        modified = hidden.clone()
        counts.clear()
        for index, mode in enumerate(modes):
            current = mask[index]
            assert int(current.sum()) == int(payload[index].shape[0]) == 1
            if mode == 'add':
                modified[index, current] = hidden[index, current] + payload[index]
            elif mode == 'replace':
                modified[index, current] = payload[index]
            else:
                raise ValueError(mode)
            counts.append(int(current.sum()))
        return replace_output_hidden(output, modified)
    handle = DECODER_LAYERS[layer].register_forward_hook(hook)
    try:
        with torch.inference_mode():
            output = adapter.model(**prepared['inputs'], use_cache=False)
    finally:
        handle.remove()
    records = score_records(prepared, output.logits)
    for record, condition, count in zip(records, conditions, counts):
        record.update({
            key: value for key, value in condition.items()
            if key not in ('payload_selected', 'target_mask')
        })
        record['patched_tokens'] = count
    return records

def score_conditions(example, conditions):
    records = []
    grouped = defaultdict(list)
    for condition in conditions:
        grouped[int(condition['layer_index'])].append(condition)
    for layer_conditions in grouped.values():
        for start in range(0, len(layer_conditions), PATCH_CONDITION_BATCH_SIZE):
            actual = layer_conditions[start:start + PATCH_CONDITION_BATCH_SIZE]
            padded = list(actual)
            while len(padded) < PATCH_CONDITION_BATCH_SIZE:
                padded.append(padded[-1])
            prepared = prepare_examples([example] * len(padded))
            records.extend(forward_intervention_chunk(
                prepared, padded
            )[:len(actual)])
            del prepared
    return records

def centered_scores(record):
    values = np.asarray(record['candidate_logprobs'], dtype=float)
    return values - values.mean()

def label_score(record, label):
    labels = list(record['candidate_labels'])
    values = np.asarray(record['candidate_logprobs'], dtype=float)
    index = labels.index(label)
    other = np.delete(values, index)
    return float(values[index] - np.logaddexp.reduce(other))

def entropy(record):
    probabilities = np.asarray(record['candidate_probabilities'], dtype=float)
    return float(-np.sum(probabilities * np.log(np.clip(probabilities, 1e-12, 1))))

def cosine_similarity(left, right):
    left = left.float().reshape(-1)
    right = right.float().reshape(-1)
    denominator = torch.linalg.vector_norm(left) * torch.linalg.vector_norm(right)
    assert float(denominator) > 0
    return float(torch.dot(left, right) / denominator)

def rescale_direction(tensor, target_norm):
    norm = selected_norm(tensor)
    assert norm > 0 and target_norm > 0
    return (tensor.float() * (float(target_norm) / norm)).to(tensor.dtype)

def unit_direction(tensor):
    return rescale_direction(tensor, 1.0).float()

def score_l49_and_capture_l55(example, payload_selected, clean_l55, metadata):
    prepared = prepare_examples([example])
    mask = final_mask(prepared)
    captured = {}
    def semantic_hook(_module, _inputs, output):
        hidden = output_hidden_tensor(output)
        payload = payload_selected.to(device=hidden.device, dtype=hidden.dtype)
        current = mask.to(hidden.device)
        modified = hidden.clone()
        modified[0, current] = hidden[0, current] + payload
        return replace_output_hidden(output, modified)
    def label_capture_hook(_module, _inputs, output):
        captured['hidden'] = output_hidden_tensor(output).detach().cpu().clone()
    semantic_handle = DECODER_LAYERS[SEMANTIC_LAYER].register_forward_hook(
        semantic_hook
    )
    label_handle = DECODER_LAYERS[LABEL_LAYER].register_forward_hook(
        label_capture_hook
    )
    try:
        with torch.inference_mode():
            output = adapter.model(**prepared['inputs'], use_cache=False)
    finally:
        semantic_handle.remove()
        label_handle.remove()
    record = score_records(prepared, output.logits)[0]
    downstream = captured['hidden'][0][mask] - clean_l55
    record.update(metadata)
    record['patched_tokens'] = int(mask.sum())
    record['downstream_l55_norm'] = selected_norm(downstream)
    return record, downstream

print('Semantic steering and downstream capture scorers ready.')


def semantic_margin(record, exact_semantic, wrong_semantic):
    semantics = list(record['candidate_semantics'])
    values = np.asarray(record['candidate_logprobs'], dtype=float)
    return float(
        values[semantics.index(exact_semantic)]
        - values[semantics.index(wrong_semantic)]
    )

def vector_projection(delta, direction):
    unit = unit_direction(direction).reshape(-1)
    return float(torch.dot(delta.float().reshape(-1), unit))

def forward_add_and_capture(example, layer, payload_selected, capture_layers):
    prepared = prepare_examples([example])
    mask = final_mask(prepared)
    captured = {}
    handles = []
    def intervention_hook(_module, _inputs, output):
        hidden = output_hidden_tensor(output)
        payload = payload_selected.to(device=hidden.device, dtype=hidden.dtype)
        current = mask.to(hidden.device)
        assert int(current.sum()) == int(payload.shape[0]) == 1
        modified = hidden.clone()
        modified[0, current] = hidden[0, current] + payload
        return replace_output_hidden(output, modified)
    handles.append(DECODER_LAYERS[layer].register_forward_hook(intervention_hook))
    for capture_layer in capture_layers:
        assert capture_layer > layer, (layer, capture_layer)
        def capture_hook(_module, _inputs, output, capture_layer=capture_layer):
            captured[capture_layer] = output_hidden_tensor(
                output
            ).detach().cpu().clone()[0]
        handles.append(
            DECODER_LAYERS[capture_layer].register_forward_hook(capture_hook)
        )
    try:
        with torch.inference_mode():
            output = adapter.model(**prepared['inputs'], use_cache=False)
    finally:
        for handle in handles:
            handle.remove()
    record = score_records(prepared, output.logits)[0]
    return record, {
        capture_layer: tensor[mask] for capture_layer, tensor in captured.items()
    }, int(mask.sum())

print('Downstream state-capture intervention helper ready.')


In [ ]:
%%cell_progress 10/17 Rebuild the frozen L48, L49, and L55 directions
cell_stage('Reconstructing the parent route without new scene outcomes')
from sklearn.cross_decomposition import PLSRegression
from sklearn.preprocessing import StandardScaler

left_units, left_raw_norms = [], []
code_units, code_raw_norms = defaultdict(list), []
for row in tqdm(direction_support, desc='L49/L55 directions', unit='family'):
    examples, mapping = build_direction_examples(row)
    scores, hidden, masks = capture_examples(
        examples, (SEMANTIC_LAYER, LABEL_LAYER)
    )
    for task in DONOR_TASKS:
        base_key, source_key = f'{task}__base', f'{task}__aligned_source'
        base_semantic = task_semantic(row['roles']['base']['features'], task)
        source_semantic = task_semantic(
            row['roles']['aligned_source']['features'], task
        )
        semantic_delta = (
            hidden[source_key][SEMANTIC_LAYER][masks[source_key]]
            - hidden[base_key][SEMANTIC_LAYER][masks[base_key]]
        )
        left_delta = semantic_delta if source_semantic == 'left' else -semantic_delta
        left_units.append(unit_direction(left_delta))
        left_raw_norms.append(selected_norm(semantic_delta))
        code_delta = (
            hidden[source_key][LABEL_LAYER][masks[source_key]]
            - hidden[base_key][LABEL_LAYER][masks[base_key]]
        )
        transition = (
            label_for(mapping, base_semantic), label_for(mapping, source_semantic)
        )
        code_units[transition].append(unit_direction(code_delta))
        code_raw_norms.append(selected_norm(code_delta))
    del hidden
    torch.cuda.empty_cache()

SEMANTIC_STEERING_NORM = float(np.median(left_raw_norms))
CODE_STEERING_NORM = float(np.median(code_raw_norms))
SEMANTIC_DIRECTION = rescale_direction(
    torch.stack(left_units).mean(dim=0), SEMANTIC_STEERING_NORM
)
CODE_DIRECTIONS = {
    transition: rescale_direction(
        torch.stack(values).mean(dim=0), CODE_STEERING_NORM
    ) for transition, values in code_units.items()
}

audit_manifest = pd.DataFrame(read_jsonl(AUDIT_OUTCOME_MANIFEST_PATH))
hidden_rows, captured_ids = [], []
for path in tqdm(
    sorted(AUDIT_OUTCOME_SHARD_DIR.glob('outcome_*.pt')),
    desc='Loading prior L48 states', unit='shard',
):
    payload = torch.load(path, map_location='cpu', weights_only=False)
    hidden_rows.append(payload['hidden'][str(UPSTREAM_LAYER)].float())
    captured_ids.extend(str(value) for value in payload['scene_ids'])
assert audit_manifest['scene_id'].astype(str).tolist() == captured_ids
X = torch.cat(hidden_rows, dim=0).numpy()
y = audit_manifest['net_torque'].to_numpy(dtype=float)
fit = audit_manifest['split'].isin(['probe_train', 'probe_dev']).to_numpy()
scaler = StandardScaler().fit(X[fit])
X_fit, y_fit = scaler.transform(X[fit]), y[fit]
pls = PLSRegression(n_components=4, scale=False, max_iter=1000)
pls.fit(X_fit, y_fit.reshape(-1, 1))
scores = pls.transform(X_fit)
latent_delta = scores[y_fit > 0].mean(axis=0) - scores[y_fit < 0].mean(axis=0)
score_centered = scores - scores.mean(axis=0, keepdims=True)
X_centered = X_fit - X_fit.mean(axis=0, keepdims=True)
decoder = np.linalg.lstsq(score_centered, X_centered, rcond=1e-5)[0]
raw_delta = (latent_delta @ decoder) * scaler.scale_
UPSTREAM_DIRECTION = torch.from_numpy(
    (raw_delta / 2).astype('float32')
).reshape(1, -1)

parent_upstream = next(
    row for row in parent_direction['upstream']
    if int(row['layer']) == UPSTREAM_LAYER and row['method'] == UPSTREAM_METHOD
)
assert abs(selected_norm(UPSTREAM_DIRECTION) - float(parent_upstream['norm'])) < 1e-4
assert abs(SEMANTIC_STEERING_NORM - PARENT_SEMANTIC_NORM) < 1e-4
direction_metadata = {
    'schema_version': 'fixed-mediator-matched-directions-1.0',
    'upstream_layer': UPSTREAM_LAYER, 'upstream_method': UPSTREAM_METHOD,
    'upstream_norm': selected_norm(UPSTREAM_DIRECTION),
    'semantic_norm': SEMANTIC_STEERING_NORM,
    'code_norm': CODE_STEERING_NORM,
    'matched_upstream_alpha': MATCHED_UPSTREAM_ALPHA,
    'parent_direction_sha256': sha256_file(PARENT_DIRECTION_PATH),
}
write_json_atomic(DIRECTION_LIBRARY_PATH, direction_metadata)
print(direction_metadata)


In [ ]:
%%cell_progress 11/17 Verify the frozen behavior-defined cohorts
cell_stage('Replaying both answer mappings without changing cohort membership')

baseline_rows = []
for row in tqdm(scene_manifest, desc='Matched cohort baselines', unit='scene'):
    examples, mappings = build_target_examples(row)
    scores, _, _ = capture_examples(examples, ())
    for variant in MAPPING_VARIANTS:
        score = scores[variant]
        baseline_rows.append({
            'match_id': row['match_id'], 'pair_id': row['pair_id'],
            'scene_id': row['scene_id'], 'cohort': row['cohort'],
            'side': row['side'], 'mapping_variant': variant,
            'baseline_status': row['baseline_status'],
            'exact_semantic': row['exact_semantic'],
            'shortcut_semantic': row['shortcut_semantic'],
            'predicted_semantic': score['predicted_semantic'],
            'correct': bool(score['predicted_semantic'] == row['exact_semantic']),
            'predicted_opposite': bool(
                score['predicted_semantic'] == row['shortcut_semantic']
            ),
            'clean_margin': semantic_margin(
                score, row['exact_semantic'], row['shortcut_semantic']
            ),
        })
baseline_frame = pd.DataFrame(baseline_rows)
write_jsonl_atomic(BASELINE_PATH, baseline_rows)
baseline_frame['expected_correct'] = baseline_frame['baseline_status'].eq('correct')
baseline_frame['replayed_status'] = np.where(
    baseline_frame['correct'], 'correct',
    np.where(baseline_frame['predicted_opposite'], 'error', 'unstable_or_balanced')
)
baseline_frame['replay_concordant'] = (
    baseline_frame['replayed_status'] == baseline_frame['baseline_status']
)
replay_by_cell = baseline_frame.groupby(
    ['cohort', 'side']
)['replay_concordant'].mean()
BASELINE_REPLAY_CONCORDANCE = float(baseline_frame['replay_concordant'].mean())
MINIMUM_CELL_REPLAY_CONCORDANCE = float(replay_by_cell.min())
BASELINE_REPLAY_GATE = bool(
    BASELINE_REPLAY_CONCORDANCE >= BASELINE_REPLAY_MIN_CONCORDANCE
    and MINIMUM_CELL_REPLAY_CONCORDANCE >= BASELINE_REPLAY_MIN_CELL_CONCORDANCE
)
baseline_summary = baseline_frame.groupby(['cohort', 'side']).agg(
    mapping_runs=('correct', 'size'), accuracy=('correct', 'mean'),
    opposite_rate=('predicted_opposite', 'mean'),
    mean_clean_margin=('clean_margin', 'mean'),
    replay_concordance=('replay_concordant', 'mean'),
).round(4)
display(baseline_summary)
print('Global / minimum-cell replay concordance:',
      BASELINE_REPLAY_CONCORDANCE, MINIMUM_CELL_REPLAY_CONCORDANCE)
print('Baseline replay gate:', BASELINE_REPLAY_GATE)
print('No scene is removed after observing replay behavior.')


In [ ]:
%%cell_progress 12/17 Run parity, matching, mapping, and norm smoke tests
cell_stage('Checking fixed route reconstruction and matched intervention geometry')

smoke_row = scene_manifest[0]
smoke_examples, smoke_mappings = build_target_examples(smoke_row)
smoke_scores, smoke_hidden, smoke_masks = capture_examples(
    smoke_examples, CAPTURE_LAYERS
)
self_errors = {}
for layer in CAPTURE_LAYERS:
    for variant in MAPPING_VARIANTS:
        record = score_conditions(smoke_examples[variant], [{
            'condition': 'self_replay', 'layer_index': layer,
            'intervention_mode': 'replace',
            'payload_selected': smoke_hidden[variant][layer][smoke_masks[variant]],
            'target_mask': smoke_masks[variant],
        }])[0]
        error = float(np.max(np.abs(
            centered_scores(record) - centered_scores(smoke_scores[variant])
        )))
        assert error < SMOKE_CENTERED_LOGPROB_TOLERANCE
        self_errors[f'{layer}:{variant}'] = error

for name, tensor in (
    ('upstream', UPSTREAM_DIRECTION * MATCHED_UPSTREAM_ALPHA),
    ('semantic', SEMANTIC_DIRECTION),
    ('code', next(iter(CODE_DIRECTIONS.values()))),
):
    random = matched_random(tensor, ('smoke', name))
    relative = abs(selected_norm(random) - selected_norm(tensor)) / selected_norm(tensor)
    assert relative < NORM_RELATIVE_TOLERANCE, (name, relative)
assert Counter(row['exact_semantic'] for row in scene_manifest) == {
    'left': 28, 'right': 28,
}
assert Counter((row['cohort'], row['side']) for row in scene_manifest) == {
    (cohort, side): MATCHED_UNITS for cohort in COHORTS for side in SIDES
}
print('Matched upstream alpha:', MATCHED_UPSTREAM_ALPHA)
print('Target L49 projection:', SEMANTIC_STEERING_NORM)
print('Maximum self-replay error:', max(self_errors.values()))


## 3. Matched causal comparison of side and error status


In [ ]:
%%cell_progress 13/17 Run frozen mediator interventions in all matched cells
cell_stage('Processing original and mirror scenes with resumable shards')

capture_identity = {
    'protocol_sha256': sha256_file(PROTOCOL_LOCK_PATH),
    'scene_manifest_sha256': sha256_file(SCENE_MANIFEST_PATH),
    'direction_sha256': sha256_file(DIRECTION_LIBRARY_PATH),
    'conditions': CONDITION_SCALES,
}
CAPTURE_SHA256 = 'sha256:' + hashlib.sha256(
    json.dumps(capture_identity, sort_keys=True).encode('utf-8')
).hexdigest()

for row in tqdm(scene_manifest, desc='Matched error-specific scenes', unit='scene'):
    shard_path = SHARD_DIR / f"{row['scene_id']}.json"
    if shard_path.is_file():
        existing = json.loads(shard_path.read_text(encoding='utf-8'))
        assert existing['capture_identity_sha256'] == CAPTURE_SHA256
        continue
    examples, mappings = build_target_examples(row)
    records = []
    exact, wrong = row['exact_semantic'], row['shortcut_semantic']
    toward_exact_sign = 1.0 if exact == 'left' else -1.0
    for variant in MAPPING_VARIANTS:
        example, mapping = examples[variant], mappings[variant]
        clean_scores, clean_hidden, clean_masks = capture_examples(
            {variant: example}, CAPTURE_LAYERS
        )
        clean = clean_scores[variant]
        clean_margin = semantic_margin(clean, exact, wrong)
        clean_l49 = clean_hidden[variant][SEMANTIC_LAYER][clean_masks[variant]]
        clean_l55 = clean_hidden[variant][LABEL_LAYER][clean_masks[variant]]
        exact_label = label_for(mapping, exact)
        wrong_label = label_for(mapping, wrong)
        code_direction = CODE_DIRECTIONS[(wrong_label, exact_label)]
        other_variant = 'reverse' if variant == 'forward' else 'forward'
        other_mapping = mappings[other_variant]
        stale_transition = (
            label_for(other_mapping, wrong), label_for(other_mapping, exact)
        )

        upstream_exact = toward_exact_sign * UPSTREAM_DIRECTION
        l49_exact = toward_exact_sign * SEMANTIC_DIRECTION
        payloads = {
            'upstream_natural': upstream_exact,
            'upstream_halfmatched': upstream_exact * CONDITION_SCALES['upstream_halfmatched'],
            'upstream_matched': upstream_exact * CONDITION_SCALES['upstream_matched'],
            'upstream_onehalfmatched': upstream_exact * CONDITION_SCALES['upstream_onehalfmatched'],
            'upstream_reverse_matched': -upstream_exact * CONDITION_SCALES['upstream_reverse_matched'],
            'upstream_random_matched': matched_random(
                upstream_exact * CONDITION_SCALES['upstream_random_matched'],
                ('paired', row['scene_id'], variant, 'upstream'),
            ),
            'l49_direct': l49_exact,
            'l49_reverse': -l49_exact,
            'l49_random': matched_random(
                l49_exact, ('paired', row['scene_id'], variant, 'l49'),
            ),
            'l55_direct': code_direction,
            'l55_stale': CODE_DIRECTIONS[stale_transition],
            'l55_random': matched_random(
                code_direction, ('paired', row['scene_id'], variant, 'l55'),
            ),
        }
        for condition, payload in payloads.items():
            if condition.startswith('upstream_'):
                layer, capture_layers = UPSTREAM_LAYER, (SEMANTIC_LAYER, LABEL_LAYER)
            elif condition.startswith('l49_'):
                layer, capture_layers = SEMANTIC_LAYER, (LABEL_LAYER,)
            else:
                layer, capture_layers = LABEL_LAYER, ()
            record, downstream, patched = forward_add_and_capture(
                example, layer, payload, capture_layers
            )
            margin = semantic_margin(record, exact, wrong)
            if SEMANTIC_LAYER in downstream:
                l49_shift = toward_exact_sign * vector_projection(
                    downstream[SEMANTIC_LAYER] - clean_l49, SEMANTIC_DIRECTION
                )
            elif condition.startswith('l49_'):
                l49_shift = toward_exact_sign * vector_projection(
                    payload, SEMANTIC_DIRECTION
                )
            else:
                l49_shift = np.nan
            if LABEL_LAYER in downstream:
                l55_shift = vector_projection(
                    downstream[LABEL_LAYER] - clean_l55, code_direction
                )
            else:
                l55_shift = np.nan
            record.update({
                'pair_id': row['pair_id'], 'scene_id': row['scene_id'],
                'source_scene_id': row['source_scene_id'],
                'orientation': row['orientation'],
                'match_id': row['match_id'], 'cohort': row['cohort'],
                'side': row['side'],
                'baseline_status': row['baseline_status'],
                'mapping_variant': variant,
                'diagnostic_cell': row['diagnostic_cell'],
                'condition': condition, 'layer_index': layer,
                'exact_semantic': exact, 'shortcut_semantic': wrong,
                'exact_label': exact_label, 'shortcut_label': wrong_label,
                'clean_margin': clean_margin,
                'margin_shift': margin - clean_margin,
                'l49_toward_exact': l49_shift,
                'l55_toward_exact_code': l55_shift,
                'rescued': bool(record['predicted_semantic'] == exact),
                'payload_norm': selected_norm(payload),
                'patched_tokens': patched,
            })
            records.append(record)
    write_json_atomic(shard_path, {
        'schema_version': 'mediator-matched-paired-shard-1.0',
        'capture_identity_sha256': CAPTURE_SHA256,
        'scene_id': row['scene_id'], 'records': records,
    })
    torch.cuda.empty_cache()
print('Completed scene shards:', len(list(SHARD_DIR.glob('*.json'))))


In [ ]:
%%cell_progress 14/17 Assemble mapping-averaged matched estimands
cell_stage('Reducing answer mappings while retaining quadruplet, cohort, and side')

all_records = []
for row in scene_manifest:
    payload = json.loads(
        (SHARD_DIR / f"{row['scene_id']}.json").read_text(encoding='utf-8')
    )
    assert payload['capture_identity_sha256'] == CAPTURE_SHA256
    all_records.extend(payload['records'])
record_frame = pd.DataFrame(all_records)
record_frame.to_csv(ANALYSIS_DIR / 'intervention_records.csv', index=False)
estimand_frame = record_frame.groupby(
    ['match_id', 'pair_id', 'scene_id', 'cohort', 'side',
     'baseline_status', 'diagnostic_cell', 'condition'], as_index=False
).agg(
    margin_shift=('margin_shift', 'mean'),
    l49_toward_exact=('l49_toward_exact', 'mean'),
    l55_toward_exact_code=('l55_toward_exact_code', 'mean'),
    rescued=('rescued', 'mean'), payload_norm=('payload_norm', 'mean'),
)
estimand_frame.to_csv(ANALYSIS_DIR / 'mapping_averaged_estimands.csv', index=False)
assert estimand_frame['match_id'].nunique() == MATCHED_UNITS
assert estimand_frame['pair_id'].nunique() == 2 * MATCHED_UNITS
display(estimand_frame.groupby(['cohort', 'side', 'condition'])[
    ['margin_shift', 'l49_toward_exact', 'l55_toward_exact_code', 'rescued']
].mean().round(4))


In [ ]:
%%cell_progress 15/17 Test side effects and the error-specific interaction
cell_stage('Bootstrapping matched quadruplets without selecting outcomes')
import gc

rng = np.random.default_rng(SEED)

def bootstrap_summary(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    assert len(values) == MATCHED_UNITS, len(values)
    draws = rng.choice(
        values, size=(BOOTSTRAP_REPLICATES, len(values)), replace=True
    ).mean(axis=1)
    return {
        'estimate': float(values.mean()),
        'ci_low': float(np.quantile(draws, ALPHA_LEVEL / 2)),
        'ci_high': float(np.quantile(draws, 1 - ALPHA_LEVEL / 2)),
        'n_matched_units': int(len(values)),
    }

def nonzero(result):
    return bool(result['ci_low'] > 0 or result['ci_high'] < 0)

def equivalent(result, margin):
    return bool(result['ci_low'] > -margin and result['ci_high'] < margin)

def matched_contrasts(condition, metric):
    current = estimand_frame[estimand_frame['condition'].eq(condition)]
    wide = current.pivot(
        index='match_id', columns=['cohort', 'side'], values=metric
    ).sort_index()
    both_gap = wide[('both_correct', 'right')] - wide[('both_correct', 'left')]
    error_gap = wide[('right_error', 'right')] - wide[('right_error', 'left')]
    right_difference = (
        wide[('right_error', 'right')] - wide[('both_correct', 'right')]
    )
    left_control = (
        wide[('right_error', 'left')] - wide[('both_correct', 'left')]
    )
    interaction = error_gap - both_gap
    return {
        'both_correct_side_gap': both_gap,
        'right_error_side_gap': error_gap,
        'right_error_minus_right_correct': right_difference,
        'left_control_difference': left_control,
        'difference_in_differences': interaction,
    }

tests = []
metrics = ('l49_toward_exact', 'l55_toward_exact_code', 'margin_shift')
for condition in CONDITION_SCALES:
    for metric in metrics:
        for contrast, values in matched_contrasts(condition, metric).items():
            if int(np.isfinite(np.asarray(values, dtype=float)).sum()) != MATCHED_UNITS:
                continue
            tests.append({
                'condition': condition, 'metric': metric, 'contrast': contrast,
                **bootstrap_summary(values),
            })
test_frame = pd.DataFrame(tests)
test_frame.to_csv(ANALYSIS_DIR / 'matched_bootstrap_tests.csv', index=False)

primary = test_frame[
    test_frame['condition'].eq('upstream_matched')
].set_index(['metric', 'contrast'])
def result(metric, contrast):
    row = primary.loc[(metric, contrast)]
    return {
        'estimate': float(row['estimate']),
        'ci_low': float(row['ci_low']), 'ci_high': float(row['ci_high']),
    }

L49_INTERACTION_MARGIN = MEDIATOR_MATCH_FRACTION * PARENT_SEMANTIC_NORM
interaction_l49 = result('l49_toward_exact', 'difference_in_differences')
interaction_l55 = result('l55_toward_exact_code', 'difference_in_differences')
interaction_answer = result('margin_shift', 'difference_in_differences')
right_l55 = result('l55_toward_exact_code', 'right_error_minus_right_correct')
right_answer = result('margin_shift', 'right_error_minus_right_correct')
left_l55 = result('l55_toward_exact_code', 'left_control_difference')
left_answer = result('margin_shift', 'left_control_difference')

gates = {
    'matching_balance': bool(MATCHING_BALANCE_GATE),
    'baseline_replay': bool(BASELINE_REPLAY_GATE),
    'l49_interaction_equivalent_zero': equivalent(
        interaction_l49, L49_INTERACTION_MARGIN
    ),
    'l55_interaction_nonzero': nonzero(interaction_l55),
    'answer_interaction_nonzero': nonzero(interaction_answer),
    'right_l55_difference_nonzero': nonzero(right_l55),
    'right_answer_difference_nonzero': nonzero(right_answer),
    'left_l55_control_equivalent': equivalent(left_l55, CODE_EQUIVALENCE_MARGIN),
    'left_answer_control_equivalent': equivalent(
        left_answer, ANSWER_EQUIVALENCE_MARGIN
    ),
}
late_interaction = gates['l55_interaction_nonzero'] or gates['answer_interaction_nonzero']
right_specific = (
    gates['right_l55_difference_nonzero'] or gates['right_answer_difference_nonzero']
)
left_stable = (
    gates['left_l55_control_equivalent'] and gates['left_answer_control_equivalent']
)
if not gates['baseline_replay']:
    PRIMARY_DECISION = 'behavior_defined_cohorts_do_not_replay_stably'
elif not gates['matching_balance']:
    PRIMARY_DECISION = 'cohorts_available_but_matching_balance_is_weak'
elif gates['l49_interaction_equivalent_zero'] and late_interaction and right_specific and left_stable:
    PRIMARY_DECISION = 'late_handoff_difference_is_specific_to_right_side_errors'
elif gates['l49_interaction_equivalent_zero'] and not late_interaction:
    PRIMARY_DECISION = 'downstream_asymmetry_is_not_error_specific_at_current_power'
elif not gates['l49_interaction_equivalent_zero']:
    PRIMARY_DECISION = 'cohort_difference_is_already_present_by_l49'
else:
    PRIMARY_DECISION = 'right_side_error_specificity_is_inconclusive'

decision = {
    'primary_decision': PRIMARY_DECISION,
    'gates': gates,
    'matching_audit': matching_audit,
    'primary_results': {
        'l49_interaction': interaction_l49,
        'l55_interaction': interaction_l55,
        'answer_interaction': interaction_answer,
        'right_l55_error_minus_correct': right_l55,
        'right_answer_error_minus_correct': right_answer,
        'left_l55_control': left_l55,
        'left_answer_control': left_answer,
    },
    'equivalence_margins': {
        'l49': L49_INTERACTION_MARGIN,
        'l55': CODE_EQUIVALENCE_MARGIN,
        'answer': ANSWER_EQUIVALENCE_MARGIN,
    },
    'claim_boundary': (
        'This is a matched three-cell identification design, not a complete '
        'side-by-correctness factorial. Behavior defines the cohorts, while '
        'all causal sites, directions, doses, and contrasts are frozen.'
    ),
}
write_json_atomic(ANALYSIS_DIR / 'error_specific_handoff_decision.json', decision)
display(test_frame[
    test_frame['condition'].eq('upstream_matched')
].round(4))
print('Gates:', gates)
print('Primary decision:', PRIMARY_DECISION)

if 'adapter' in globals():
    del adapter
gc.collect()
torch.cuda.empty_cache()


In [ ]:
%%cell_progress 16/17 Audit matching, mappings, interventions, and replay
cell_stage('Checking the partial-factorial claim boundary')

assert len(match_manifest) == MATCHED_UNITS
assert len(scene_manifest) == 4 * MATCHED_UNITS
assert len({row['pair_id'] for row in scene_manifest}) == 2 * MATCHED_UNITS
assert len({row['scene_id'] for row in scene_manifest}) == 4 * MATCHED_UNITS
assert Counter((row['cohort'], row['side']) for row in scene_manifest) == {
    (cohort, side): MATCHED_UNITS for cohort in COHORTS for side in SIDES
}
assert len(baseline_frame) == len(scene_manifest) * len(MAPPING_VARIANTS)
assert record_frame['scene_id'].nunique() == 4 * MATCHED_UNITS
assert set(record_frame['mapping_variant']) == set(MAPPING_VARIANTS)
assert record_frame['patched_tokens'].eq(1).all()
assert set(record_frame['condition']) == set(CONDITION_SCALES)
assert record_frame.groupby(
    ['scene_id', 'mapping_variant', 'condition']
).size().eq(1).all()

norm_groups = (
    ('upstream_matched', 'upstream_reverse_matched', 'upstream_random_matched'),
    ('l49_direct', 'l49_reverse', 'l49_random'),
    ('l55_direct', 'l55_stale', 'l55_random'),
)
norm_rows = []
for keys, current in record_frame.groupby(['scene_id', 'mapping_variant']):
    values_by_condition = current.set_index('condition')['payload_norm']
    for index, conditions in enumerate(norm_groups):
        values = values_by_condition.loc[list(conditions)].to_numpy(dtype=float)
        relative_range = float((values.max() - values.min()) / values.mean())
        norm_rows.append({
            'scene_id': keys[0], 'mapping_variant': keys[1],
            'control_family': index, 'relative_range': relative_range,
        })
norm_frame = pd.DataFrame(norm_rows)
assert float(norm_frame['relative_range'].max()) < NORM_RELATIVE_TOLERANCE
norm_frame.to_csv(ANALYSIS_DIR / 'norm_audit.csv', index=False)

audit_summary = {
    'matched_quadruplets': MATCHED_UNITS,
    'physical_pairs': 2 * MATCHED_UNITS,
    'physical_scenes': 4 * MATCHED_UNITS,
    'factorial_cells': {
        f'{cohort}:{side}': MATCHED_UNITS
        for cohort in COHORTS for side in SIDES
    },
    'complete_2x2_claimed': False,
    'causal_results_used_for_selection': False,
    'answer_mapping_variants': len(MAPPING_VARIANTS),
    'baseline_replay_concordance': BASELINE_REPLAY_CONCORDANCE,
    'minimum_cell_replay_concordance': MINIMUM_CELL_REPLAY_CONCORDANCE,
    'baseline_replay_gate': BASELINE_REPLAY_GATE,
    'fixed_parent_route': [UPSTREAM_LAYER, UPSTREAM_METHOD],
    'matched_upstream_alpha': MATCHED_UPSTREAM_ALPHA,
    'matching_balance_gate': MATCHING_BALANCE_GATE,
    'maximum_absolute_matching_smd': MAXIMUM_ABSOLUTE_SMD,
    'diagnostic_match_rate': DIAGNOSTIC_MATCH_RATE,
    'maximum_self_replay_centered_error': max(self_errors.values()),
    'maximum_control_norm_relative_range': float(
        norm_frame['relative_range'].max()
    ),
}
write_json_atomic(ANALYSIS_DIR / 'error_specific_handoff_audit.json', audit_summary)
print(audit_summary)


In [ ]:
%%cell_progress 17/17 Save the error-specific handoff figure and report
cell_stage('Writing the matched three-cell evidence card')
import matplotlib.pyplot as plt

primary_tests = test_frame[test_frame['condition'].eq('upstream_matched')]
def test_row(metric, contrast):
    return primary_tests[
        primary_tests['metric'].eq(metric)
        & primary_tests['contrast'].eq(contrast)
    ].iloc[0]

fig, axes = plt.subplots(1, 4, figsize=(18, 4.8))
accuracy = baseline_frame.groupby(['cohort', 'side'])['correct'].mean()
x = np.arange(len(COHORTS))
width = 0.34
for index, side in enumerate(SIDES):
    axes[0].bar(
        x + (index - 0.5) * width,
        [accuracy.loc[(cohort, side)] for cohort in COHORTS],
        width, label=side.title(),
        color=('#2563EB' if side == 'left' else '#D97706'),
    )
axes[0].set_xticks(x, ['Both correct', 'Right error'])
axes[0].set_ylim(0, 1.05)
axes[0].set_ylabel('Baseline accuracy')
axes[0].set_title('A. Behavior-defined cells')
axes[0].legend(frameon=False)

plot_specs = (
    ('l49_toward_exact', 'L49 toward correct side', 'B. Early physical coordinate'),
    ('l55_toward_exact_code', 'L55 toward current answer code', 'C. Late response state'),
    ('margin_shift', 'Correct-vs-opposite log-odds shift', 'D. Answer effect'),
)
for axis, (metric, ylabel, title) in zip(axes[1:], plot_specs):
    rows = [
        test_row(metric, 'both_correct_side_gap'),
        test_row(metric, 'right_error_side_gap'),
    ]
    estimates = np.asarray([row['estimate'] for row in rows], dtype=float)
    lower = estimates - np.asarray([row['ci_low'] for row in rows], dtype=float)
    upper = np.asarray([row['ci_high'] for row in rows], dtype=float) - estimates
    axis.bar(
        ['Both correct', 'Right error'], estimates,
        color=['#0F766E', '#DC2626'],
        yerr=np.vstack([lower, upper]), capsize=4,
    )
    axis.axhline(0, color='black', linewidth=0.8)
    axis.set_ylabel(ylabel)
    axis.set_title(title)
fig.suptitle(PRIMARY_DECISION, y=1.03)
fig.tight_layout()
figure_path = FIGURE_DIR / 'rightside_error_specific_handoff.png'
fig.savefig(figure_path, dpi=220, bbox_inches='tight')
plt.show()

def interval(metric, contrast):
    row = test_row(metric, contrast)
    return f"{row['estimate']:.5f} [{row['ci_low']:.5f}, {row['ci_high']:.5f}]"

lines = [
    '# Qwen-27B Right-Side Error-Specific Handoff', '',
    f'**Primary decision:** `{PRIMARY_DECISION}`', '',
    '## Design', '',
    f'- Matched quadruplets: {MATCHED_UNITS}',
    '- Cohort 1: both left and right mirror orientations are stable-correct.',
    '- Cohort 2: left is stable-correct and right is stable-error.',
    '- All layers, directions, doses, mappings, and contrasts were frozen.',
    '- This is a matched three-cell design, not a complete 2×2 factorial.', '',
    '## Matching', '',
    f'- Diagnostic-cell match rate: {DIAGNOSTIC_MATCH_RATE:.3f}',
    f'- Maximum absolute standardized difference: {MAXIMUM_ABSOLUTE_SMD:.3f}',
    f'- Precommitted balance gate: `{MATCHING_BALANCE_GATE}`', '',
    '## Baseline replay', '',
    f'- Global concordance: {BASELINE_REPLAY_CONCORDANCE:.3f}',
    f'- Minimum cell concordance: {MINIMUM_CELL_REPLAY_CONCORDANCE:.3f}',
    f'- Precommitted replay gate: `{BASELINE_REPLAY_GATE}`', '',
    '## Matched-upstream interaction', '',
    f"- L49: {interval('l49_toward_exact', 'difference_in_differences')}",
    f"- L55: {interval('l55_toward_exact_code', 'difference_in_differences')}",
    f"- Answer: {interval('margin_shift', 'difference_in_differences')}", '',
    '## Right-side error versus right-side correct', '',
    f"- L55: {interval('l55_toward_exact_code', 'right_error_minus_right_correct')}",
    f"- Answer: {interval('margin_shift', 'right_error_minus_right_correct')}", '',
    '## Left-correct control', '',
    f"- L55: {interval('l55_toward_exact_code', 'left_control_difference')}",
    f"- Answer: {interval('margin_shift', 'left_control_difference')}", '',
    '## Claim boundary', '',
    (
        'The analysis can separate right-side correct from right-side error '
        'while controlling left-correct behavior and matched geometry. It '
        'cannot estimate the missing left-error cell or claim a complete '
        'side-by-correctness interaction.'
    ), '',
]
report_path = ANALYSIS_DIR / 'RESULT_CARD.md'
report_path.write_text('\n'.join(lines), encoding='utf-8')
run_complete = {
    'status': 'complete',
    'completed_at_utc': datetime.now(timezone.utc).isoformat(),
    'model_id': MODEL_ID, 'revision': MODEL_REVISION,
    'primary_decision': PRIMARY_DECISION,
    'matched_quadruplets': MATCHED_UNITS,
    'matching_balance_gate': MATCHING_BALANCE_GATE,
    'baseline_replay_gate': BASELINE_REPLAY_GATE,
    'protocol_sha256': sha256_file(PROTOCOL_LOCK_PATH),
    'match_manifest_sha256': sha256_file(MATCH_MANIFEST_PATH),
    'scene_manifest_sha256': sha256_file(SCENE_MANIFEST_PATH),
    'decision_sha256': sha256_file(
        ANALYSIS_DIR / 'error_specific_handoff_decision.json'
    ),
    'figure': str(figure_path), 'report': str(report_path),
}
write_json_atomic(OUTPUT_ROOT / 'RUN_COMPLETE.json', run_complete)
print('Saved:', figure_path)
print('Saved:', report_path)
print('RUN_COMPLETE:', OUTPUT_ROOT / 'RUN_COMPLETE.json')
